In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Реакция и наблюдение

Если при подготовке пробы меняется химическая форма, спектр и хроматограмма уже относятся к другой смеси. Начнём с элементарного контроля стехиометрии, затем рассмотрим условный переход кислота→лактон B. Для линии A опубликованная циклизация ксантинов даёт контекст реакции, но её выход не является константой скорости.

Сначала вычислите формулы и массы кислоты/лактона B, не меняя исходные записи. Затем решите условную ODE k=.02 min^-1 на 0…120 min и сравните с аналитическим решением. Постройте обе концентрации и проверьте баланс в каждой точке.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M09-L01','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
from rdkit import Chem
from rdkit.Chem import Descriptors,rdMolDescriptors
forms=[Chem.MolFromSmiles(compound(n)['smiles']) for n in ['atorvastatin','atorvastatin_lactone']]
print([rdMolDescriptors.CalcMolFormula(m) for m in forms])
assert abs(Descriptors.MolWt(forms[0])-Descriptors.MolWt(forms[1])-18.015)<1e-6


['C33H35FN2O5', 'C33H33FN2O4']


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
import numpy as np
from scipy.integrate import solve_ivp
time=np.linspace(0,120,121);k=.02;initial=1e-4
sol=solve_ivp(lambda t,y:[-k*y[0],k*y[0]],[0,120],[initial,0],t_eval=time,rtol=1e-9,atol=1e-13)
assert sol.success
analytic=initial*np.exp(-k*time)
assert np.max(abs(sol.y[0]-analytic))<1e-11 and np.allclose(sol.y.sum(axis=0),initial,atol=1e-12)
fig,ax=plt.subplots();ax.plot(time,sol.y[0],label='acid model');ax.plot(time,sol.y[1],label='lactone model')
ax.set(xlabel='Time / min',ylabel='Simulated concentration / mol L^-1');ax.legend();publish_plot(fig,'kinetics.png')


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'kind':'simulation','k_min_inverse':k,'maximum_solver_error':float(np.max(abs(sol.y[0]-analytic))),
        'actual_B_stability_measurement':'NOT_RUN'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M09-L01','A',result,ROOT,inputs=['practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M09-L01\20261118T141615852473Z-A-d371d6489b25470b9330998af1c49ddd.json


Совпадение ODE с аналитикой проверяет реализацию. Оно не измеряет стабильность B и не доказывает механизм. Для константы скорости нужен временной эксперимент, который станет задачей следующего урока.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?